# Reprodução de uma run de modelagem

Este notebook foi gerado a partir de um template. Ele lê o `run.yaml` da pasta da run, treina somente o algoritmo e a janela definidos ali e mostra as métricas.

O notebook não substitui o `manifest.json`: ele é uma forma didática e reproduzível de inspecionar uma run específica.

## 1. Carregar a configuração da run

O caminho abaixo é preenchido quando o notebook é criado. Abra o JupyterLab na raiz do projeto para que o dataset e a pasta da run sejam encontrados.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import yaml
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, TargetEncoder
import matplotlib.pyplot as plt

RUN_DIR = Path("runs/000001")
if not RUN_DIR.exists():
    RUN_DIR = Path(".")
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = RUN_DIR.parents[1]
SPEC = yaml.safe_load((RUN_DIR / "run.yaml").read_text(encoding="utf-8"))
DATA_PATH = ROOT / SPEC["dataset"]["arquivo"]
FAIXAS = {0: "Pontual ou antecipado", 1: "Atraso inferior a 15 min", 2: "Atraso de 15 a 30 min", 3: "Atraso superior a 30 até 45 min", 4: "Atraso superior a 45 até 60 min", 5: "Atraso superior a 60 min"}
SPEC["nome"], SPEC["modelo"]["algoritmo"], DATA_PATH

## 2. Carregar dados e aplicar a janela temporal

As datas finais são exclusivas. A coluna `data_referencia` controla a separação, mas não entra como variável do modelo.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df["data_referencia"] = pd.to_datetime(df["data_referencia"])
split = SPEC["divisao"]
treino = df[df["data_referencia"].between(split["treino_inicio"], split["treino_fim_exclusivo"], inclusive="left")].copy()
avaliacao = df[df["data_referencia"].between(split["avaliacao_inicio"], split["avaliacao_fim_exclusivo"], inclusive="left")].copy()
categoricas = SPEC["variaveis"]["categoricas"]
numericas = SPEC["variaveis"]["numericas"]
features = categoricas + numericas
print("Treino:", treino.shape, "Avaliação:", avaliacao.shape)

## 3. Construir o modelo definido no YAML

Cada algoritmo usa o mesmo pré-processamento registrado no `run.yaml`. Target Encoding é ajustado dentro do pipeline, somente com o treino.

In [ ]:
algoritmo = SPEC["modelo"]["algoritmo"]
parametros = SPEC["modelo"]["parametros"]
if algoritmo == "baseline":
    modelo = DummyClassifier(**parametros)
elif algoritmo == "regressao_logistica":
    pre = ColumnTransformer([
        ("categoricas", Pipeline([("imputacao", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), categoricas),
        ("numericas", SimpleImputer(strategy="median"), numericas),
    ])
    modelo = Pipeline([("preprocessamento", pre), ("classificador", LogisticRegression(**parametros))])
elif algoritmo == "random_forest":
    pre = Pipeline([("imputacao", SimpleImputer(strategy="most_frequent")), ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])
    modelo = Pipeline([("preprocessamento", pre), ("classificador", RandomForestClassifier(**parametros))])
elif algoritmo == "hist_gradient_boosting":
    pre = ColumnTransformer([
        ("categoricas", Pipeline([("imputacao", SimpleImputer(strategy="most_frequent")), ("target_encoding", TargetEncoder(target_type="multiclass", smooth=20.0, cv=5, random_state=42))]), categoricas),
        ("numericas", SimpleImputer(strategy="median"), numericas),
    ])
    modelo = Pipeline([("preprocessamento", pre), ("classificador", HistGradientBoostingClassifier(**parametros))])
else:
    raise ValueError(f"Algoritmo desconhecido: {algoritmo}")
modelo

## 4. Treinar e avaliar a run

A execução abaixo reproduz o treinamento descrito no `run.yaml`. Ela mostra as métricas e a matriz de confusão, mas não substitui os arquivos de resultado da run.

In [ ]:
modelo.fit(treino[features], treino["faixa_atraso"])
predicoes = modelo.predict(avaliacao[features])
metricas = {
    "accuracy": accuracy_score(avaliacao["faixa_atraso"], predicoes),
    "balanced_accuracy": balanced_accuracy_score(avaliacao["faixa_atraso"], predicoes),
    "macro_f1": f1_score(avaliacao["faixa_atraso"], predicoes, average="macro"),
}
print(metricas)
print(classification_report(avaliacao["faixa_atraso"], predicoes, labels=list(FAIXAS), target_names=list(FAIXAS.values()), zero_division=0))

## 5. Matriz de confusão

In [ ]:
matrix = confusion_matrix(avaliacao["faixa_atraso"], predicoes, labels=list(FAIXAS))
display = ConfusionMatrixDisplay(matrix, display_labels=list(FAIXAS.values()))
fig, ax = plt.subplots(figsize=(10, 8))
display.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.tight_layout()